### Import Required Dependencies

In [15]:
#Project Setup

# --- Core Libraries ---
import pandas as pd
import numpy as np
import warnings
import logging


# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Path and File Handling ---
from pathlib import Path

# --- Data Wrangling and Utilities ---
from functools import reduce
from sklearn.preprocessing import MinMaxScaler

# --- Visualization ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# --- Warning ---
warnings.filterwarnings('ignore')



## Project Work flow 

In [16]:
# --- Define Project Directory (relative path for loading and saving) ---
data_dir = Path("data")     # Directory for input data
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# --- Configure Logging ---
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    filename=log_dir / "project.log",
    level=logging.INFO,  # Change to DEBUG for more detail
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode='w'  # overwrite log on each run
)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("[%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)


print("✅ Environment ready. Data and output paths set.")

✅ Environment ready. Data and output paths set.


In [ ]:
#Execute utility functions
# This will run the utility functions defined in utils.ipynb
%run utils.ipynb

from utils import (
    standardize_column_names,
    filter_to_county_level,
    log_duplicate_attributes,
    extract_common_year_from_columns,
    subset_columns_by_year,
    nca_counties
)

ModuleNotFoundError: No module named 'utils'

ModuleNotFoundError: No module named 'utils'

## Section 2A: 

### Section 2B: Load, Clean, Pivot, and Merge National Datasets (with State Info)


In [ ]:
# Section 2B: Load, Clean, Pivot, and Merge National Datasets (with State Info)

# Define file paths
complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

# Utility to clean column names
def standardize_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ \-]+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

# Filter out state-level rows (e.g., Arkansas State Total)
def filter_to_county_level(df):
    for fips_col in ['fips', 'fips_code', 'fipstxt']:
        if fips_col in df.columns:
            df = df[df[fips_col].astype(str).str[-3:] != '000']
    if 'county' in df.columns:
        df = df[~df['county'].str.contains("state|total|arkansas", case=False, na=False)]
    return df

# Load and clean all datasets
complete_data = {}
state_lookup = None

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)

        # Rename consistent fields
        df.rename(columns={'fips_code': 'fips', 'fipstxt': 'fips', 'area_name': 'county'}, inplace=True)

        # Clean key fields
        if 'county' in df.columns:
            df['county'] = df['county'].str.strip().str.lower()
        if 'attribute' in df.columns:
            df['attribute'] = df['attribute'].str.strip().str.lower()

        # Preserve state info from education dataset
        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.strip().str.lower()
            state_lookup['state'] = state_lookup['state'].str.strip().str.upper()

        # Log duplicates
        if df.duplicated(subset=['county', 'attribute']).any():
            logging.warning(f"⚠️ {key} has duplicate county-attribute pairs.")

        complete_data[key] = df
        logging.info(f"✅ Loaded & cleaned {filename}: {df.shape[0]} rows | Columns: {df.columns.tolist()}")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")

# Pivot all datasets to wide format
try:
    edu_wide = complete_data['edu'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    pop_wide = complete_data['pop'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    poverty_wide = complete_data['poverty'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    unemp_wide = complete_data['unemp'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    logging.info("✅ Pivoted all datasets to wide format.")
except Exception as e:
    logging.error(f"❌ Pivoting failed: {e}")

# Merge all pivoted datasets
try:
    df_full = reduce(lambda left, right: pd.merge(left, right, on='county', how='outer'),
                     [edu_wide, pop_wide, poverty_wide, unemp_wide])
    logging.info(f"✅ Merged dataset shape: {df_full.shape}")
except Exception as e:
    logging.error(f"❌ Merge failed: {e}")

# Attach preserved state info
try:
    df_full = df_full.reset_index()
    df_full = df_full.merge(state_lookup, on='county', how='left')
    df_full['state'] = df_full['state'].str.strip().str.upper()
    df_full.set_index('county', inplace=True)
    logging.info(f"📌 Attached state info. Final shape: {df_full.shape}")
except Exception as e:
    logging.error(f"❌ Failed to attach state info: {e}")

# Save to CSV
df_full.to_csv(output_dir / "us_county_merged.csv")
logging.info("💾 Saved full merged dataset to /outputs/us_county_merged.csv")


## Section 2B: Subset and Prepare Arkansas / NCA Counties

In [ ]:
# Step 1: Reset index if needed
df_full = df_full.reset_index() if df_full.index.name == 'county' else df_full

# Step 2: Filter to Arkansas only
df_full['state'] = df_full['state'].str.strip().str.upper()
df_ar = df_full[df_full['state'] == 'AR'].copy()

# Step 3: Clean the county names *on df_ar only*
df_ar['county'] = (
    df_ar['county']
    .str.lower()
    .str.replace(" county", "", regex=False)
    .str.replace(", ar", "", regex=False)
    .str.strip()
)

# Step 4: Set index to cleaned county names
df_ar.set_index('county', inplace=True)

# Step 5: Define NCA counties
nca_counties = [
    'baxter','cleburne','fulton', 'independence', 'izard', 'jackson', 'marion',
    'searcy', 'sharp', 'stone', 'van buren', 'white','woodruff'
]

# Step 6: Subset NCA from AR
df_nca = df_ar[df_ar.index.isin(nca_counties)].copy()

# Step 7: Log and Save
logging.info(f"📌 Subset: Arkansas counties = {df_ar.shape[0]}")
logging.info(f"📌 Subset: NCA counties = {df_nca.shape[0]}")

df_ar.to_csv(output_dir / "arkansas_counties.csv")
df_nca.to_csv(output_dir / "nca_counties.csv")
logging.info("💾 Saved Arkansas and NCA subsets to /outputs")


### Step 3: Inspect Dataset

### Step 4: Visualize Distributions (Histograms & KDE)

### Step 5: Correlation Matrix and Heatmap

### Step 6: Scatter Plots for Key Relationships

### Step 7: Identify Outlier Counties with Boxplots

### Step 8: Log-Transform Population (Optional)
If the population is skewed: